In [1]:
#2.2 编程题:不使用深度学习框架的高级 API（仅使用 Tensor 基础算子如 torch.matmul等），纯 NumPy 或 PyTorch 从零实现一个多分类（使用 Fashion-MNIST数据集）的单隐藏层 MLP。
#1.手动初始化隐藏层参数 W1, b1 和输出层参数 W2, b2（提示：使用正态分布随机初始化）。
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
# 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.view(-1)) # 拉平成向量
])
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
# 验证数据形状
for X, y in train_loader:
    print(f"Batch shape: {X.shape}") # [64, 784]
    break

#2.实现 ReLU 激活函数的前向传播：max(0, x)
class MyMLP:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) * 0.01
        self.b1 = torch.zeros(hidden_dim)
        self.W2 = torch.randn(hidden_dim, output_dim) * 0.01
        self.b2 = torch.zeros(output_dim)
        self.params = [self.W1, self.b1, self.W2, self.b2]

    def relu(self, x):
        return torch.maximum(x, torch.tensor(0.0))

    def forward(self, X):
        self.hidden = self.relu(X @ self.W1 + self.b1)
        logits = self.hidden @ self.W2 + self.b2
        return logits
# 初始化模型
model = MyMLP(input_dim=784, hidden_dim=256, output_dim=10)

#3.实现带有 Softmax 的交叉熵损失函数。
def cross_entropy_loss(logits, y_true):
    # 数值稳定版 Softmax
    logits = logits - torch.max(logits, dim=1, keepdim=True)[0]
    exp_logits = torch.exp(logits)
    probs = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)    
    # 交叉熵
    n = y_true.shape[0]
    loss = -torch.log(probs[torch.arange(n), y_true]).mean()
    return loss

#4.编写训练循环，通过小批量随机梯度下降（Mini-batch SGD）手动更新参数。
def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()
optimizer_lr = 0.1
epochs = 3
for epoch in range(epochs):
    total_loss = 0
    for X_batch, y_batch in train_loader:
        #前向
        hidden = model.relu(X_batch @ model.W1 + model.b1)
        logits = hidden @ model.W2 + model.b2

        #损失
        logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
        exp_logits = torch.exp(logits_stable)
        probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)

        n = y_batch.shape[0]
        loss = -torch.log(probs[torch.arange(n), y_batch]).mean()
        total_loss += loss.item()

        #反向传播手动
        # 1. 输出层梯度
        dlogits = probs.clone()
        dlogits[torch.arange(n), y_batch] -= 1
        dlogits /= n

        # 2. W2, b2
        dW2 = hidden.T @ dlogits
        db2 = dlogits.sum(dim=0)

        # 3. 回传隐藏层
        dh = dlogits @ model.W2.T
        dh[hidden <= 0] = 0  # ReLU 梯度

        # 4. W1, b1
        dW1 = X_batch.T @ dh
        db1 = dh.sum(dim=0)

        #参数更新（SGD）
        with torch.no_grad():
            model.W2 -= optimizer_lr * dW2
            model.b2 -= optimizer_lr * db2
            model.W1 -= optimizer_lr * dW1
            model.b1 -= optimizer_lr * db1

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(train_loader):.4f}")

Batch shape: torch.Size([64, 784])
Epoch 1, Avg Loss: 0.5680
Epoch 2, Avg Loss: 0.4031
Epoch 3, Avg Loss: 0.3627


In [ ]:
# 3.2 MLP 基础上加入 L2 正则化和 Dropout
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.view(-1))
])
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# 实现 Dropout 层
def dropout_layer(X, dropout, is_training=True):
    if not is_training or dropout == 0:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

# 自定义 MLP，包含 L2 和 Dropout（手动实现）
class MyMLP_Reg:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = torch.randn(input_dim, hidden_dim) * 0.01
        self.b1 = torch.zeros(hidden_dim)
        self.W2 = torch.randn(hidden_dim, output_dim) * 0.01
        self.b2 = torch.zeros(output_dim)
        self.params = [self.W1, self.b1, self.W2, self.b2]

    def relu(self, x):
        return torch.maximum(x, torch.tensor(0.0))

    def forward(self, X, dropout_rate, is_training=True):
        h = X @ self.W1 + self.b1
        h = self.relu(h)
        if is_training:
            h = dropout_layer(h, dropout_rate, is_training)
        logits = h @ self.W2 + self.b2
        return logits

def cross_entropy_loss_with_l2(logits, y_true, model, lambda_l2):
    # 交叉熵
    logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
    exp_logits = torch.exp(logits_stable)
    probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)
    n = y_true.shape[0]
    ce_loss = -torch.log(probs[torch.arange(n), y_true]).mean()
    # L2 正则化（权重衰减，不含偏置）
    l2_penalty = (model.W1.pow(2).sum() + model.W2.pow(2).sum()) * lambda_l2
    return ce_loss + l2_penalty

# 训练函数（支持 Dropout 和 L2）
def train_mlp_reg(model, train_loader, test_loader, epochs, lr, dropout_rate, lambda_l2, is_training=True):
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for X_batch, y_batch in train_loader:
            # 前向
            logits = model.forward(X_batch, dropout_rate, is_training=True)
            loss = cross_entropy_loss_with_l2(logits, y_batch, model, lambda_l2)
            total_loss += loss.item()

            # 反向传播手动
            n = y_batch.shape[0]
            # softmax 梯度
            logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
            exp_logits = torch.exp(logits_stable)
            probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)
            dlogits = probs.clone()
            dlogits[torch.arange(n), y_batch] -= 1
            dlogits /= n

            # 隐藏层输出（带 Dropout 前的结果，用于梯度计算）
            h = model.relu(X_batch @ model.W1 + model.b1)
            # 计算梯度时需考虑 Dropout mask 的影响
            if is_training and dropout_rate > 0:
                mask = (torch.rand(h.shape) > dropout_rate).float()
                h_dropped = mask * h / (1.0 - dropout_rate)
            else:
                h_dropped = h

            dW2 = h_dropped.T @ dlogits
            db2 = dlogits.sum(dim=0)

            dh = dlogits @ model.W2.T
            if is_training and dropout_rate > 0:
                dh = dh * mask / (1.0 - dropout_rate)  # 反向传播也要缩放
            dh[h <= 0] = 0  # ReLU 梯度

            dW1 = X_batch.T @ dh
            db1 = dh.sum(dim=0)

            # 参数更新（SGD + 权重衰减）
            with torch.no_grad():
                model.W2 = model.W2 * (1 - lr * lambda_l2) - lr * dW2
                model.b2 -= lr * db2
                model.W1 = model.W1 * (1 - lr * lambda_l2) - lr * dW1
                model.b1 -= lr * db1

            # 计算准确率
            pred = logits.argmax(dim=1)
            correct += (pred == y_batch).sum().item()
            total += n

        avg_loss = total_loss / len(train_loader)
        acc = correct / total
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Train Acc: {acc:.4f}")

# 评估函数（无 Dropout）
def evaluate(model, test_loader):
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits = model.forward(X_batch, dropout_rate=0, is_training=False)
            pred = logits.argmax(dim=1)
            correct += (pred == y_batch).sum().item()
            total += y_batch.shape[0]
    return correct / total

# 对比实验：设计高维多项式拟合或使用极少样本训练一个复杂的 MLP，绘制并对比：1) 无正则化、2) 有权重衰减、3) 有 Dropout 三种情况下的训练和验证误差曲线（Loss Curve）。
# 为了快速演示，取前 200 个样本作为训练集，其余作为验证集
small_train_data = torch.utils.data.Subset(train_data, range(200))
small_train_loader = DataLoader(small_train_data, batch_size=64, shuffle=True)
val_data = torch.utils.data.Subset(train_data, range(200, 500))
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)

def run_experiment(use_l2=False, use_dropout=False, l2_lambda=0.001, dropout_rate=0.5):
    model = MyMLP_Reg(784, 256, 10)
    epochs = 10
    lr = 0.1
    lambda_l2 = l2_lambda if use_l2 else 0.0
    dr = dropout_rate if use_dropout else 0.0
    train_mlp_reg(model, small_train_loader, val_loader, epochs, lr, dr, lambda_l2)
    val_acc = evaluate(model, val_loader)
    print(f"Validation Accuracy: {val_acc:.4f}")
    return val_acc

print("=== No Regularization ===")
run_experiment(use_l2=False, use_dropout=False)

print("\n=== With Weight Decay (L2) ===")
run_experiment(use_l2=True, use_dropout=False, l2_lambda=0.001)

print("\n=== With Dropout ===")
run_experiment(use_l2=False, use_dropout=True, dropout_rate=0.5)

=== No Regularization ===
Epoch 1, Loss: 2.2663, Train Acc: 0.1500
Epoch 2, Loss: 2.1427, Train Acc: 0.2750
Epoch 3, Loss: 1.8897, Train Acc: 0.4000
Epoch 4, Loss: 1.6309, Train Acc: 0.5250
Epoch 5, Loss: 1.4449, Train Acc: 0.4700
Epoch 6, Loss: 1.2010, Train Acc: 0.5950
Epoch 7, Loss: 1.1684, Train Acc: 0.5800
Epoch 8, Loss: 1.2432, Train Acc: 0.6050
Epoch 9, Loss: 1.0047, Train Acc: 0.5900
Epoch 10, Loss: 0.7453, Train Acc: 0.6850
Validation Accuracy: 0.6033

=== With Weight Decay (L2) ===
Epoch 1, Loss: 2.2845, Train Acc: 0.2950
Epoch 2, Loss: 2.0988, Train Acc: 0.2800
Epoch 3, Loss: 1.9002, Train Acc: 0.3200
Epoch 4, Loss: 1.6672, Train Acc: 0.4500
Epoch 5, Loss: 1.4242, Train Acc: 0.5150
Epoch 6, Loss: 1.3751, Train Acc: 0.5500
Epoch 7, Loss: 1.2020, Train Acc: 0.5900
Epoch 8, Loss: 1.0135, Train Acc: 0.6800
Epoch 9, Loss: 1.0928, Train Acc: 0.6700
Epoch 10, Loss: 0.9627, Train Acc: 0.6900
Validation Accuracy: 0.5533

=== With Dropout ===
Epoch 1, Loss: 2.2683, Train Acc: 0.2000
E

0.5433333333333333

In [ ]:
# 4.2 模拟梯度消失/爆炸，验证不同初始化
import torch
import torch.nn as nn

#构建深层网络：使用 PyTorch 的高级 API (nn.Sequential) 构建一个20 层的深层全连接网络，隐藏层宽度设为 256
class DeepNet(nn.Module):
    def __init__(self, layers=20, width=256, activation='sigmoid'):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(layers):
            self.layers.append(nn.Linear(width, width))
        self.activation = activation

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.activation == 'sigmoid':
                x = torch.sigmoid(x)
            elif self.activation == 'relu':
                x = torch.relu(x)
        return x

# 注册 hook 获取梯度范数
def get_grad_norms(model):
    norms = []
    for name, param in model.named_parameters():
        if param.grad is not None:
            norms.append(param.grad.norm().item())
    return norms

# 模拟梯度消失/爆炸：全部激活函数采用 Sigmoid，权重采用普通高斯分布初始化（如 nn.init.normal_(m.weight, mean=0, std=1)），输入随机数据，观察并打印前几层和后几层的梯度范数（Gradient Norm），验证梯度消失。. 激活函数采用 ReLU，权重采用较大的初值（如 std=10），观察是否发生 NaN（梯度爆炸或数值溢出）。
model_vanish = DeepNet(activation='sigmoid')
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=1)
model_vanish.apply(init_weights)

x = torch.randn(64, 256)  # 小批量
out = model_vanish(x)
loss = out.sum()  # dummy loss
loss.backward()
norms = get_grad_norms(model_vanish)
print("Sigmoid + N(0,1) 梯度范数（前几层 vs 后几层）:")
print(f"第一层梯度范数: {norms[0]:.6f}, 最后一层梯度范数: {norms[-1]:.6f}")
# 预期：梯度消失，前几层极小

#激活函数采用 ReLU，权重采用较大的初值（如 std=10），观察是否发生 NaN（梯度爆炸或数值溢出）。
model_explode = DeepNet(activation='relu')
def init_large(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=10)
model_explode.apply(init_large)
x = torch.randn(64, 256)
out = model_explode(x)
loss = out.sum()
loss.backward()
print("\nReLU + N(0,10) 梯度范数（观察是否有 NaN）:")
try:
    norms = get_grad_norms(model_explode)
    print(f"第一层梯度范数: {norms[0]:.6f}, 最后一层梯度范数: {norms[-1]:.6f}")
except Exception as e:
    print("出现 NaN 或数值溢出:", e)

#修复与验证：使用 Xavier 初始化（nn.init.xavier_uniform_）结合ReLU（或 LeakyReLU），再次打印各层的梯度分布，观察其是否稳定在合理区间（例如 [1e-6, 1e3]）。
model_fixed = DeepNet(activation='relu')
def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
model_fixed.apply(init_xavier)
x = torch.randn(64, 256)
out = model_fixed(x)
loss = out.sum()
loss.backward()
norms = get_grad_norms(model_fixed)
print("\nXavier + ReLU 梯度范数（稳定区间）:")
print(f"第一层梯度范数: {norms[0]:.6f}, 最后一层梯度范数: {norms[-1]:.6f}")
# 预期梯度范数在 [1e-6, 1e3] 之间

Sigmoid + N(0,1) 梯度范数（前几层 vs 后几层）:
第一层梯度范数: 10667.619141, 最后一层梯度范数: 58.527359

ReLU + N(0,10) 梯度范数（观察是否有 NaN）:
第一层梯度范数: nan, 最后一层梯度范数: 1024.000000

Xavier + ReLU 梯度范数（稳定区间）:
第一层梯度范数: 2.550557, 最后一层梯度范数: 744.975830


In [ ]:
# 5.2 模拟协变量偏移，使用加权线性回归校正
import torch
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

#人工数据集构造：训练集 P：从正态分布 N (−1, 1) 中采样 1000 个特征 x，标签 y = 2x + ϵ（ϵ 为小噪声）
# 训练集 P: x ~ N(-1,1), y = 2x + noise
torch.manual_seed(42)
np.random.seed(42)

n_train = 1000
x_train = np.random.normal(-1, 1, n_train)
noise = np.random.normal(0, 0.1, n_train)
y_train = 2 * x_train + noise
X_train = x_train.reshape(-1, 1)

#测试集 Q：从正态分布 N (2, 1) 中采样 500 个特征 x（此时发生了明显的协变量偏移）。
n_test = 500
x_test = np.random.normal(2, 1, n_test)
y_test = 2 * x_test + np.random.normal(0, 0.1, n_test)
X_test = x_test.reshape(-1, 1)

#基线模型：用一个简单的线性回归模型直接在训练集 P 上训练，并在测试集 Q 上评估，记录均方误差（MSE）。
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
print(f"Baseline MSE (no correction): {mse_baseline:.6f}")

#偏移校正实现：编写一个逻辑回归分类器，将训练集 P 的样本标记为类别 0，测试集 Q 的样本标记为类别 1 。
# 将训练集和测试集混合，标记训练集为0，测试集为1
X_combined = np.vstack([X_train, X_test])
y_combined = np.array([0]*n_train + [1]*n_test)

clf = LogisticRegression()
clf.fit(X_combined, y_combined)

# 计算每个训练样本的权重
# (a) 预测每个训练样本属于测试集的概率 P(test|x)
prob_test = clf.predict_proba(X_train)[:, 1]  # 类别1的概率

# (b) 计算权重 w_i ∝ P(test|x_i) / P(train|x_i)
prob_train = 1 - prob_test
weights = prob_test / (prob_train + 1e-8)          # 避免除零
weights = weights / weights.sum() * n_train       # 归一化（可选）

#加权模型训练：使用这些权重重新训练线性回归模型（加权最小二乘法），并再次在测试集 Q 上评估。对比校正前后的测试 MSE，验证校正效果。
# 加权最小二乘法手动实现
X_train_w = np.hstack([np.ones((n_train, 1)), X_train])  # 添加截距列
W = np.diag(weights)
beta = np.linalg.inv(X_train_w.T @ W @ X_train_w) @ (X_train_w.T @ W @ y_train)
y_pred_weighted = beta[0] + beta[1] * X_test.flatten()
mse_weighted = mean_squared_error(y_test, y_pred_weighted)

print(f"Corrected MSE (with sample weights): {mse_weighted:.6f}")
print(f"Improvement: {mse_baseline - mse_weighted:.6f}")

Baseline MSE (no correction): 0.010182
Corrected MSE (with sample weights): 0.024026
Improvement: -0.013843
